# 新角度盒需圖測試(消除離群).ipynb

步態畢業專題研究筆記。此檔保留探索步驟，已清除輸出、執行資訊及 Colab 中繼資料。

包含獨立實驗與尚未統一的座標定義，不保證由上而下可執行；使用整合流程請從根目錄 README 開始。舊輸出不作為本次驗證結果。


In [ ]:
import os
import math
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files
import io


In [ ]:
def calculate_angle_change(x1, y1, x2, y2, x3, y3):
    a_x = x2 - x1
    a_y = y2 - y1
    b_x = x3 - x2
    b_y = y3 - y2
    dot_product = a_x * b_x + a_y * b_y
    magnitude_a = np.hypot(a_x, a_y)
    magnitude_b = np.hypot(b_x, b_y)
    if magnitude_a * magnitude_b == 0:
        return 0
    cos_angle = dot_product / (magnitude_a * magnitude_b)
    cos_angle = max(-1, min(1, cos_angle))
    return np.arccos(cos_angle) * (180 / np.pi)


In [ ]:
def remove_outliers(angles):
    flat_angles = [angle for angle_list in angles for angle in angle_list]
    Q1 = pd.Series(flat_angles).quantile(0.25)
    Q3 = pd.Series(flat_angles).quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    cleaned_data = []
    for file_angles in angles:
        cleaned_angles = [angle for angle in file_angles if lower_bound <= angle <= upper_bound]
        cleaned_data.append(cleaned_angles)

    return cleaned_data


In [ ]:
category_results = {}


In [ ]:
for category in range(1, 5):
    print(f"請上傳第 {category} 類的CSV文件")
    uploaded_files = files.upload()

    category_angles = []
    file_names = []


    for file_name in uploaded_files.keys():
        df = pd.read_csv(io.BytesIO(uploaded_files[file_name]))

        if 'kpts' in df.columns:
            kpts_columns = df['kpts'].str.split(',', expand=True)
            kpts_columns.columns = [f'kpt_{i+1}' for i in range(kpts_columns.shape[1])]
            df = df.drop(columns=[col for col in df.columns if col.startswith('kpt_')], errors='ignore')
            df = df.join(kpts_columns)

        required_kpts = ['kpt_31', 'kpt_32', 'kpt_34', 'kpt_35', 'kpt_40', 'kpt_41', 'kpt_44', 'kpt_45']
        if not all(kpt in df.columns for kpt in required_kpts):
            print(f"文件 {file_name} 缺少必要的關鍵點數據，請檢查文件內容。")
            continue

        x11 = df['kpt_31'].astype(float)
        y11 = df['kpt_32'].astype(float)
        x12 = df['kpt_34'].astype(float)
        y12 = df['kpt_35'].astype(float)
        x14 = df['kpt_40'].astype(float)
        y14 = df['kpt_41'].astype(float)
        x15 = df['kpt_44'].astype(float)
        y15 = df['kpt_45'].astype(float)
        x18 = (x11 + x12) / 2
        y18 = (y11 + y12) / 2

        angles = []
        for i in range(len(x18)):
            angle_change_value = calculate_angle_change(x14.iloc[i], y14.iloc[i], x18.iloc[i], y18.iloc[i], x15.iloc[i], y15.iloc[i])
            angles.append(angle_change_value)

        category_angles.append(angles)
        file_names.append(file_name)


    cleaned_angles = remove_outliers(category_angles)
    category_results[category] = cleaned_angles


In [ ]:
data = [np.hstack(category_results[cat]) for cat in range(1, 5)]


In [ ]:
plt.figure(figsize=(10, 6))


plt.boxplot(data, patch_artist=True, boxprops=dict(facecolor='skyblue'))


plt.title('Comparison of Angle Changes for All Categories', fontsize=16)
plt.xlabel('Category', fontsize=14)
plt.ylabel('Angle Change (degrees)', fontsize=14)
plt.xticks([1, 2, 3, 4], ['Category 1', 'Category 2', 'Category 3', 'Category 4'])
plt.grid(True)

plt.tight_layout()
plt.show()


In [ ]:
q1_values = []
q2_values = []
q3_values = []


In [ ]:
for category in range(1, 5):
    flat_angles = [angle for angle_list in category_results[category] for angle in angle_list]
    q1 = pd.Series(flat_angles).quantile(0.25)
    q2 = pd.Series(flat_angles).quantile(0.5)
    q3 = pd.Series(flat_angles).quantile(0.75)

    q1_values.append(q1)
    q2_values.append(q2)
    q3_values.append(q3)


In [ ]:
plt.figure(figsize=(10, 18))
plt.subplot(3, 1, 1)
plt.plot(range(1, 5), q1_values, label='Q1', marker='o')
plt.title('Q1 Angle Change Line Plot', fontsize=14)
plt.xlabel('Category', fontsize=12)
plt.ylabel('Q1 Angle Change (degrees)', fontsize=12)
plt.xticks([1, 2, 3, 4], ['Category 1', 'Category 2', 'Category 3', 'Category 4'])
plt.grid(True)


In [ ]:
plt.subplot(3, 1, 2)
plt.plot(range(1, 5), q2_values, label='Q2', marker='o')
plt.title('Q2 Angle Change Line Plot', fontsize=14)
plt.xlabel('Category', fontsize=12)
plt.ylabel('Q2 Angle Change (degrees)', fontsize=12)
plt.xticks([1, 2, 3, 4], ['Category 1', 'Category 2', 'Category 3', 'Category 4'])
plt.grid(True)


In [ ]:
plt.subplot(3, 1, 3)
plt.plot(range(1, 5), q3_values, label='Q3', marker='o')
plt.title('Q3 Angle Change Line Plot', fontsize=14)
plt.xlabel('Category', fontsize=12)
plt.ylabel('Q3 Angle Change (degrees)', fontsize=12)
plt.xticks([1, 2, 3, 4], ['Category 1', 'Category 2', 'Category 3', 'Category 4'])
plt.grid(True)


In [ ]:
lt.tight_layout()
plt.show()
